In [92]:
import numpy as np
import tracemalloc
import time
import random

# Frequency Estimation and Heavy Hitters

## Helper Functions

Below, we provide some helper functions that will be useful for this assignment.

First, we give a function for measuring the time and memory usage of function calls. To measure the time and memory usage of a function call ``myFunc(x,y,z)`` use the following code: ``measure_time_and_memory(myFunc, x, y, z)``.

In [93]:
def measure_time_and_memory(func, *args, **kwargs):
    tracemalloc.start()
    start_time = time.perf_counter()

    result = func(*args, **kwargs)

    end_time = time.perf_counter()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return end_time - start_time, peak / 10 ** 6, result

We also provide an implementation of the simple tabulation hash function which returns a 32-bit integer hash value.

In [94]:
def simple_tab32(H):
    return lambda x: _simple_tab32(x, H)


def _simple_tab32(x: int, H: np.ndarray) -> int:
    h = 0
    for i in range(4):
        c = x & 0xFF  # Extract the lowest 8 bits
        h ^= H[i, c]
        x >>= 8
    return h

 You can use these hash functions as follows:

In [95]:
H = np.random.randint(0, 2 ** 32, size=(4, 256), dtype=np.uint32)
myHashFunc = simple_tab32(H)
print(myHashFunc(5))
print(myHashFunc(10))

3076983013
1834486911


You also find a function ``sample_zipf_like`` which allows for sampling from Zipfian distributions.

In [96]:
def sample_zipf_like(a, n, size):
    """
    Sample from a Zipf-like distribution over {1, ..., n} with exponent a.
    
    Parameters:
        a (float): Exponent (e.g., 0.5 for 1/sqrt(i))
        n (int): Maximum rank
        size (int): Number of samples to draw
        
    Returns:
        np.ndarray: Array of samples
    """
    weights = np.array([1 / (i ** a) for i in range(1, n + 1)])
    probabilities = weights / weights.sum()
    return np.random.choice(np.arange(1, n + 1), size=size, p=probabilities)

An example call is as follows:

In [97]:
a = 0.75
n = 10000
numSamples = 1000000

sample = sample_zipf_like(a, n, numSamples)

## Implementation of Frequency Estimation Data Structures

In [98]:
class DictionaryBaseline:
    def __init__(self):
        self.counters = {}

    def update(self, x, v):
        self.counters[x] = self.counters.get(x, 0) + v

    def query(self, x):
        return self.counters.get(x, 0)

In [99]:
class MisraGries:
    def __init__(self, maxNumCounters):
        # Initialize a set of counters that is initially empty
        self.counters = {}
        self.maxNumCounters = maxNumCounters

    def update(self, x, v):
        if v < 0: raise ValueError("MisraGries does not allow negative numbers")

        # if there exists a counter for x_i then
        if x in self.counters:
            # Increase the counter of x_i by v_i
            self.counters[x] += v

        else:
            # Create a new counter for x_i and initialize it with v_i
            self.counters[x] = v

            # if there are more than 1/ε counters then
            if len(self.counters) > self.maxNumCounters:
                # Find the value vmin of the smallest counter
                v_min = min(self.counters.values())

                for x_key in list(self.counters.keys()):
                    # Decrease all counters by v_min
                    self.counters[x_key] -= v_min
                    if self.counters[x_key] <= 0:
                        # Remove all counters that are 0
                        self.counters.pop(x_key)

    def query(self, x):
        # if there exists a counter for x then return the value of the counter for x else return 0
        return self.counters.get(x, 0)

In [100]:
class CountMinSketch:
    def __init__(self, numCols: int, numRows: int, hashSeed = 42):

        # numRows ~= k
        self.numRows = numRows
        # numCols ~= l
        self.numCols = numCols

        np.random.seed(hashSeed)

        # h_1, ..., h_k ← independently sample k pairwise independent hash functions mapping from U to [l]
        self.hashFunctions = []
        for _ in range(numRows): # 1, ..., k
            H = np.random.randint(0, 2 ** 32, size=(4, 256), dtype=np.uint32)
            self.hashFunctions.append(simple_tab32(H))

        # C ← an all-zeroes matrix of size k × ℓ
        self.C = np.zeros(shape=(numRows, numCols), dtype=np.int64)

    def update(self, x, v):
        if v < 0: raise ValueError("CountMinSketch does not allow negative numbers")

        for j, h_j in enumerate(self.hashFunctions): # j = 1, ..., k
            col = h_j(x) % self.numCols
            self.C[j, col] += v

    def query(self, x):
        return min(
            self.C[j, h_j(x) % self.numCols]
            for j, h_j in enumerate(self.hashFunctions) # j = 1, ..., k
        )


## Experimental Evaluation

In [101]:
# Testing Code:
# can and probably should be removed
test_data = [(1, 1), (1, 1), (1, 2), (2, 1), (2, 2), (1, 10), (100, 100), (200, 1), (100, 1000)]

dictBL = DictionaryBaseline()
mg = MisraGries(maxNumCounters=2)
cms = CountMinSketch(numCols=3, numRows=2)

for x, v in test_data:
    dictBL.update(x, v)
    mg.update(x, v)
    cms.update(x, v)

print("\nDictionaryBaseline:")
print(dictBL.query(1))  # 14
print(dictBL.query(2))  # 3
print(dictBL.query(100))  # 1100
print(dictBL.query(200))  # 1
print(dictBL.query(999))  # 0

print("\nMisraGries:")
print(mg.query(1))  # 10
print(mg.query(2))  # 0
print(mg.query(100))  # 1096
print(mg.query(200))  # 0

print("\nCountMinSketch:")
print(cms.query(1))  # varies depending on random hashes / hashes seed
print(cms.query(2))
print(cms.query(100))
print(cms.query(200))




DictionaryBaseline:
14
3
1100
1
0

MisraGries:
10
0
1096
0

CountMinSketch:
17
17
1101
1
